# 🌍 Multilingual IT Support Desk: One Structured Call vs a Translation Chain

## Learning Objectives
In this notebook, you will learn:
1. **Tickets in ten languages** - A support queue where the customer's language is known, so language detection can be checked
2. **Design A: one structured call per ticket** - Detect the language, categorize, translate and answer in a single call that returns a typed record
3. **Design B: a four-step chain** - Detect → translate to English → answer in English → translate back, one LLM call per step
4. **Choosing between them** - Compare language accuracy, replies actually written in the customer's language, LLM calls and measured wall time

## Where this fits
Prompt chaining (Module 05) splits a task into steps; structured output (`04_langchain_agents_and_middleware/03_structured_output.ipynb`) gets several fields from one call. Both can build the same support desk. This notebook builds it both ways on the same tickets and measures the difference instead of guessing.

## Prerequisites
- Structured output and LCEL chaining from Modules 04–05 (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: ticket triage, translation and replies
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🎫 Part 1: The Support Queue

The desk receives IT tickets in many languages. For each ticket it must produce:



```
orig_msg: The original customer message
orig_lang: Detected language of the customer message e.g. Spanish
category: 1-2 word describing the category of the problem
trans_msg: Translated customer message in English
response: Response to the customer in orig_lang
trans_response: Response to the customer in English
```


The queue below combines two sets of tickets; `EXPECTED_LANG` records the language each one is written in.

In [ ]:
# ============================================================================
# TICKETS + THEIR LANGUAGES
# ============================================================================
tickets = [
    "Não consigo sincronizar meus contatos com o telefone. Sempre recebo uma mensagem de falha.",
    "Ho problemi a stampare i documenti da remoto. Il lavoro non viene inviato alla stampante di rete.",
    "プリンターのトナーを交換しましたが、印刷品質が低下しています。サポートが必要です。",
    "Я не могу войти в систему учета времени, появляется сообщение об ошибке. Мне нужна помощь.",
    "Internet bağlantım çok yavaş ve bazen tamamen kesiliyor. Yardım eder misiniz?",
    "Не могу установить обновление безопасности. Появляется код ошибки. Помогите, пожалуйста."
] + [
    "I can't access my email. It keeps showing an error message. Please help.",
    "Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?",
    "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
    "我无法访问公司的网站。每次都显示错误信息。请帮忙解决。"
]

EXPECTED_LANG = ["Portuguese", "Italian", "Japanese", "Russian", "Turkish", "Russian",
                 "English", "Spanish", "French", "Chinese"]

for lang, t in zip(EXPECTED_LANG, tickets):
    print(f"{lang:<11} {t[:70]}")

---
## 🅰️ Part 2: Design A — One Structured Call per Ticket

All six fields come back from a single call as an `ITSupportResponse`. `.batch()` sends the ten tickets concurrently; the cell measures the wall time.

In [ ]:
# ============================================================================
# DESIGN A: ONE STRUCTURED CALL
# ============================================================================
import time

import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ITSupportResponse(BaseModel):
    orig_msg: str = Field(description="The original customer IT support query message")
    orig_lang: str = Field(description="Detected language of the customer message e.g. Spanish")
    category: str = Field(description="1-2 word describing the category of the problem")
    trans_msg: str = Field(description="Translated customer IT support query message in English")
    response: str = Field(description="Response to the customer in their original language - orig_lang")
    trans_response: str = Field(description="Response to the customer in English")


support_prompt = ChatPromptTemplate.from_template(
    """Act as an Information Technology (IT) customer support agent.
    Process the IT support message below.

    Customer IT support message:
    {it_support_msg}"""
)
single_call = support_prompt | llm.with_structured_output(ITSupportResponse)

start = time.perf_counter()
design_a = [r.model_dump() for r in single_call.batch([{"it_support_msg": t} for t in tickets])]
time_a = time.perf_counter() - start

df_a = pd.DataFrame(design_a)
print(f"⏱️ Design A: {len(tickets)} tickets, {len(tickets)} LLM calls, {time_a:.1f} s\n")
print(df_a[["orig_lang", "category", "trans_msg"]].to_string(max_colwidth=70))

---
## 🅱️ Part 3: Design B — A Four-Step Translation Chain

The same job as four prompts, each doing one thing. `RunnablePassthrough.assign` adds each step's output to the running dictionary, so later steps can use earlier results (the reply is translated back into `orig_lang` from step 1).

```
orig_msg → [detect language] → orig_lang → [translate to English] → trans_msg
         → [answer in English] → trans_response → [translate back] → response
```

In [ ]:
# ============================================================================
# DESIGN B: FOUR CHAINED PROMPTS
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def step(template: str):
    return ChatPromptTemplate.from_template(template) | llm | StrOutputParser()


# Chain 1: Detect customer message language
detect = step("""
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Output the language of the message in one word only, e.g. Spanish

  Customer Message:
  ```{orig_msg}```
""")
# Chain 2: Translate customer message to English
translate_in = step("""
  Act as a customer support agent.
  For the customer message and customer message language delimited below by triple backticks,
  Translate the customer message from the customer message language to English
  if customer message language is not in English,
  else return back the original customer message.

  Customer Message:
  ```{orig_msg}```
  Customer Message Language:
  ```{orig_lang}```
""")
# Chain 3: Generate a resolution response in English
answer = step("""
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Generate an appropriate resolution response in English.

  Customer Message:
  ```{trans_msg}```
""")
# Chain 4: Translate the resolution back to the customer's language
translate_out = step("""
  Act as a customer support agent.
  For the customer resolution response and target language delimited below by triple backticks,
  Translate the customer resolution response message from English to the target language
  if target language is not in English,
  else return back the original customer resolution response.

  Customer Resolution Response:
  ```{trans_response}```
  Target Language:
  ```{orig_lang}```
""")

four_step_chain = (
    RunnablePassthrough.assign(orig_lang=detect)
    | RunnablePassthrough.assign(trans_msg=translate_in)
    | RunnablePassthrough.assign(trans_response=answer)
    | RunnablePassthrough.assign(response=translate_out)
)

start = time.perf_counter()
design_b = four_step_chain.batch([{"orig_msg": t} for t in tickets])
time_b = time.perf_counter() - start

df_b = pd.DataFrame(design_b)
print(f"⏱️ Design B: {len(tickets)} tickets, {4 * len(tickets)} LLM calls, {time_b:.1f} s\n")
print(df_b[["orig_lang", "trans_msg"]].to_string(max_colwidth=70))

---
## ⚖️ Part 4: Comparing the Two Designs

Three checks, all in code:
- **Language detected** — does `orig_lang` name the ticket's real language?
- **Reply in the customer's language** — for non-English tickets, the reply must differ from its English version (an untranslated reply is identical)
- **Clean text** — no leftovers from the prompts in what the customer or the next step sees: the triple-backtick delimiters, or a preamble such as "The customer message in English is:"
- **Cost and time** — LLM calls and the wall time measured above

In [ ]:
# ============================================================================
# SCORE BOTH DESIGNS ON THE SAME TICKETS
# ============================================================================
def lang_ok(expected: str, detected: str) -> bool:
    detected = detected.lower()
    return expected.lower() in detected or (expected == "Chinese" and "mandarin" in detected)


def score(rows: list[dict]) -> dict:
    lang = sum(lang_ok(e, r["orig_lang"]) for e, r in zip(EXPECTED_LANG, rows))
    foreign = [(e, r) for e, r in zip(EXPECTED_LANG, rows) if e != "English"]
    translated = sum(r["response"].strip() != r["trans_response"].strip() for _, r in foreign)
    clean = sum(not leaks(r) for r in rows)
    return {"language detected": f"{lang}/{len(rows)}",
            "reply translated": f"{translated}/{len(foreign)}",
            "clean text": f"{clean}/{len(rows)}"}


def leaks(r: dict) -> bool:
    """True if prompt scaffolding leaked into the translation or the reply."""
    texts = [r["trans_msg"], r["response"]]
    return any("```" in t or t.lower().lstrip().startswith(("the customer message", "translation:"))
               or "customer message in english" in t.lower() for t in texts)


comparison = pd.DataFrame({
    "A: one structured call": {**score(design_a), "LLM calls": len(tickets), "wall time (s)": round(time_a, 1)},
    "B: four-step chain": {**score(design_b), "LLM calls": 4 * len(tickets), "wall time (s)": round(time_b, 1)},
})
print(comparison.to_string())

for name, rows in [("A", design_a), ("B", design_b)]:
    for e, r in zip(EXPECTED_LANG, rows):
        if not lang_ok(e, r["orig_lang"]):
            print(f"❌ Design {name}: expected {e}, detected {r['orig_lang']!r}")
        if leaks(r):
            print(f"❌ Design {name} ({e}): leaked text → {r['trans_msg'][:60]!r} / {r['response'][:40]!r}")

In [ ]:
# ============================================================================
# ONE TICKET SIDE BY SIDE
# ============================================================================
i = EXPECTED_LANG.index("Japanese")
print(f"Ticket ({EXPECTED_LANG[i]}): {tickets[i]}\n")
for name, r in [("A", design_a[i]), ("B", design_b[i])]:
    print(f"--- Design {name} ---\nreply:   {r['response'][:250]}\nEnglish: {r['trans_response'][:250]}\n")

Read the table rather than assuming. On a queue like this both designs usually detect every language. The differences are elsewhere: Design B makes four times as many LLM calls and its steps run one after another for each ticket, and its free-text steps can leak scaffolding (a code fence copied from the prompt's backticks, "The customer message in English is: …") into the next step and into the reply — a class of error that structured output does not have. What the chain buys is **control**: each step can be tested, cached (translations), logged or given a different model (a cheap model to detect language, a stronger one to answer), and a wrong translation is visible as its own intermediate value. Start with Design A; move a step out into its own prompt when you need to inspect, reuse or upgrade it.

---
## 📝 Summary

In this notebook, we learned:

### 1. A checkable support queue
- Recording each ticket's real language turns "it seems to work" into a score

### 2. Design A: one structured call
- One call returns every field as a typed record; cheapest and simplest

### 3. Design B: a four-step chain
- Detect → translate → answer → translate back; every intermediate value is visible and each step can be changed independently

### 4. Choosing between them
- Compare on the same inputs: accuracy, whether replies are really translated, LLM calls and measured time — then pick the simplest design that passes

### Next Steps
- **`08_audience_tailored_writer.ipynb`**: one source document, several outputs for different audiences — with format and faithfulness checked in code